In [ ]:
# === 1. Import thư viện ===
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import ast  # để chuyển chuỗi list -> list thật

# === 2. Đọc dữ liệu ===
df = pd.read_csv("/content/sample_data/question_answers.csv")

# Giải mã cột answer (vì đang lưu dạng chuỗi list)
df['answer'] = df['answer'].apply(lambda x: ast.literal_eval(x))

# Tạo dữ liệu phân loại question / answer
rows = []
for _, row in df.iterrows():
    # thêm dòng question
    rows.append({"content": row['question'], "label": "question"})
    # thêm từng đáp án
    for ans in row['answer']:
        rows.append({"content": ans, "label": "answer"})

data = pd.DataFrame(rows)

# Map label thành số
label_map = {"question": 0, "answer": 1}
data["label_id"] = data["label"].map(label_map)

# Chia tập train/test
train_df, test_df = train_test_split(data, test_size=0.2, random_state=42, stratify=data['label_id'])

# === 3. Chuẩn bị tokenizer ===
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch['content'], truncation=True, padding='max_length', max_length=128)

# === 4. Chuẩn bị dataset ===
train_dataset = Dataset.from_pandas(train_df[['content', 'label_id']])
test_dataset = Dataset.from_pandas(test_df[['content', 'label_id']])
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

# Đổi tên cột label
train_dataset = train_dataset.rename_column("label_id", "labels")
test_dataset = test_dataset.rename_column("label_id", "labels")

# Định dạng dữ liệu cho torch
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# === 5. Load mô hình ===
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(label_map))

# === 6. Cấu hình huấn luyện ===
training_args = TrainingArguments(
    output_dir="./model_output",
    do_eval=True,          # tương đương evaluation_strategy="epoch"
    save_total_limit=2,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    logging_dir="./logs",
    logging_steps=50,
    fp16=True,
)



# === 7. Huấn luyện ===
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

# === 8. Lưu mô hình ===
trainer.save_model("/content/bert_question_answer_classifier")

print("✅ Fine-tune hoàn tất. Mô hình đã lưu tại /content/bert_question_answer_classifier")
print("Label map:", label_map)


In [ ]:
from transformers import pipeline
from docx import Document
import pandas as pd

# === 1. Load mô hình fine-tune ===
model_path = "/content/bert_question_answer_classifier"
clf = pipeline("text-classification", model=model_path, tokenizer="bert-base-uncased", return_all_scores=False)

# === 2. Đọc file docx ===
def extract_paragraphs_from_docx(docx_path):
    doc = Document(docx_path)
    paragraphs = []
    for para in doc.paragraphs:
        text = para.text.strip()
        if text:
            paragraphs.append(text)
    return paragraphs

# === 3. Phân loại và gom nhóm ===
def classify_and_group(docx_path, output_csv_path):
    paragraphs = extract_paragraphs_from_docx(docx_path)
    results = []

    # Thêm ánh xạ label
    label_mapping = {"LABEL_0": "question", "LABEL_1": "answer"}

    # Bước 1: phân loại từng đoạn
    classified = []
    for text in paragraphs:
        try:
            pred = clf(text)[0]
            pred["label"] = label_mapping.get(pred["label"], pred["label"])  # ánh xạ
            classified.append({"text": text, "label": pred["label"], "score": pred["score"]})
        except Exception:
            classified.append({"text": text, "label": "error", "score": 0})

    # Bước 2: gom nhóm
    grouped = []
    current_question = None
    current_answers = []

    for item in classified:
        text = item["text"]
        label = item["label"]

        if label.lower() == "question":
            if current_question and current_answers:
                grouped.append({
                    "question": current_question,
                    "answer": current_answers
                })
            current_question = text
            current_answers = []

        elif label.lower() == "answer":
            current_answers.append(text)

    if current_question and current_answers:
        grouped.append({
            "question": current_question,
            "answer": current_answers
        })

    # Bước 3: Xuất CSV
    if not grouped:
        print("⚠️ Không tìm thấy câu hỏi/đáp án nào. Kiểm tra lại model hoặc label mapping.")
    else:
        df = pd.DataFrame(grouped)
        df["answer"] = df["answer"].apply(lambda x: str(x))
        df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")
        print(f"✅ Hoàn tất! File CSV được lưu tại: {output_csv_path}")
        return df

# === 4. Chạy thử ===
input_docx = "/content/sample_data/22_Thpt_Px_Thpt_Bd-VanDao.docx"
output_csv = "/content/sample_data/question_answers_predicted.csv"

df = classify_and_group(input_docx, output_csv)
if df is not None:
    print(df.head())
